# ML (v3 recipe) + DL (CNN ens) BLEND — OOF weight search + per-file balanced threshold


# Cross-Subject EEG Motor-Imagery Decoding — v3 (v2 recipe + per-file balanced threshold)

In [ ]:
import os, warnings, re
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
np.set_printoptions(suppress=True)

data_root = "/kaggle/input/competitions/low-cost-motor-imagery-decoding-for-rehab-cross-subject"
train_dir = os.path.join(data_root, "Training set")
test_dir = os.path.join(data_root, "Cross participant test set")
print(len(os.listdir(train_dir)), "train files;", len(os.listdir(test_dir)), "test files")

In [ ]:
import numpy as np
import pandas as pd
import glob, os, re
from scipy import signal

FS = 250.0
CHANNELS = ['Fz', 'C3', 'Cz', 'C4', 'PO7', 'Pz', 'PO8', 'Oz']
CUE_LEN = 5.0
BASE_LEN = 3.0

_NOTCH_CACHE = [None]
def notch(x):
    if _NOTCH_CACHE[0] is None:
        _NOTCH_CACHE[0] = signal.iirnotch(50.0 / (FS / 2.0), 30.0)
    b, a = _NOTCH_CACHE[0]
    return signal.filtfilt(b, a, x, axis=0)

def car(x):
    return x - x.mean(axis=1, keepdims=True)

def read_epochs(path, labeled=True):
    df = pd.read_csv(path, low_memory=False)
    x = df[CHANNELS].to_numpy(np.float64)
    m = df['Marker_val'].astype(str)
    x = car(x)
    x = notch(x)
    if labeled:
        cue_idx = np.flatnonzero(m.isin(['rest', 'move']).to_numpy())
        labels = np.array([1 if v == 'move' else 0 for v in m.iloc[cue_idx]])
    else:
        cue_idx = np.flatnonzero((m == 'cue_start').to_numpy())
        labels = None
    tcue = int(CUE_LEN * FS)
    tbase = int(BASE_LEN * FS)
    cues, bases = [], []
    for i in cue_idx:
        s, e = i, i + tcue
        if e > len(x):
            continue
        cues.append(x[s:e])
        b0 = max(0, i - tbase)
        bases.append(x[b0:i][-tbase:])
    cues = np.stack(cues)
    bases = np.stack(bases)
    return cues, bases, labels


In [ ]:
"""Feature extractors (ML-track compliant: no neural networks)."""
import numpy as np
from scipy import signal
from sklearn.covariance import LedoitWolf

BANDS = [(4, 8), (8, 12), (12, 16), (16, 20), (20, 25), (25, 30)]
BANDS7 = [(4, 6), (6, 8), (8, 10), (10, 12), (12, 16), (16, 20), (20, 30)]
BUTTER = {}
def _butter(l, h, order=4):
    key = (float(l), float(h))
    if key not in BUTTER:
        nyq = FS / 2
        BUTTER[key] = signal.butter(order, [max(0.01, l / nyq), min(0.99, h / nyq)], btype='bandpass')
    return BUTTER[key]

def band_filter(x, l, h):
    b, a = _butter(l, h)
    axis = 1 if x.ndim == 3 else 0
    return signal.filtfilt(b, a, x, axis=axis)

def band_power_features(cue, base, bands=BANDS, off=0.0):
    if off > 0:
        k = int(off * FS)
        cue = cue[:, k:, :]
    """Log band power, cue absolute + relative to baseline. (N, 4*nbands)"""
    N = cue.shape[0]
    feats = []
    for (l, h) in bands:
        cf = band_filter(cue, l, h)
        bf = band_filter(base, l, h)
        pc = np.log(np.maximum((cf ** 2).mean(axis=1), 1e-12))  # (N, 8)
        pb = np.log(np.maximum((bf ** 2).mean(axis=1), 1e-12))
        feats.append(pc)
        feats.append(np.log(np.maximum((cf ** 2).mean(axis=1), 1e-12) / np.maximum((bf ** 2).mean(axis=1), 1e-12)))
    return np.concatenate(feats, axis=1)

def _spdm(cue):
    """Spatio-temporal covariance (N, 8, 8)."""
    return np.array([np.cov(x.T) for x in cue])

def _matrix_log(A):
    w, v = np.linalg.eigh(A)
    return v @ np.diag(np.log(np.maximum(w, 1e-12))) @ v.T

def riemannian_features(cue, bands=BANDS, ref=None, shrink=True, off=0.0):
    if off > 0:
        k = int(off * FS)
        cue = cue[:, k:, :]
    """FB-RTS: per-band band-filtered covariances projected onto log-Euclidean
    tangent space at reference mean. Returns (feats, ref)."""
    est = LedoitWolf()
    covs = []
    for (l, h) in bands:
        cf = band_filter(cue, l, h)
        covs.append(np.array([est.fit(x).covariance_ for x in cf]))
    if ref is None:
        ref = np.stack([np.mean(c, axis=0) for c in covs])
    feats = []
    for ci, r in zip(covs, ref):
        Ginv_sqrt = _matrix_op(r, -0.5)
        f = []
        for C in ci:
            W = Ginv_sqrt @ C @ Ginv_sqrt
            L = _matrix_log(W)
            idx = np.triu_indices(8)
            v = L[idx]
            v[1:] *= np.sqrt(2.0)
            f.append(v)
        feats.append(np.stack(f))
    return np.concatenate(feats, axis=1), ref

def _matrix_op(A, p):
    w, v = np.linalg.eigh(A)
    return v @ np.diag(w ** p) @ v.T

def csp_features(cue, y, bands=((8,12),(12,16)), k=4, off=0.0):
    if off > 0:
        k2 = int(off * FS)
        cue = cue[:, k2:, :]
    """CSP spatial filters per band; diagonal-bandpower features. Returns (feats, filters)."""
    est = LedoitWolf()
    filters = []
    feats = []
    for (l, h) in bands:
        cf = band_filter(cue, l, h)
        C1 = np.mean([np.cov(x.T) for x in cf[y == 1]], axis=0) if np.sum(y == 1) > 2 else np.eye(8)
        C0 = np.mean([np.cov(x.T) for x in cf[y == 0]], axis=0) if np.sum(y == 0) > 2 else np.eye(8)
        C1, C0 = est.fit(C1).covariance_, est.fit(C0).covariance_
        w, V = np.linalg.eigh(np.linalg.pinv(C0) @ C1)
        order = np.argsort(w)
        V = V[:, order]
        F = np.concatenate([V[:, -k:], V[:, :k]], axis=1)  # (8, 2k)
        filters.append(F)
        proj = [x @ F for x in cf]
        d = np.log(np.maximum(np.array([np.diag(np.cov(p.T)) for p in proj]), 1e-12))
        feats.append(d)
    return np.concatenate(feats, axis=1), filters

def csp_transform(cue, filters, bands=((8,12),(12,16)), off=0.0):
    if off > 0:
        k2 = int(off * FS)
        cue = cue[:, k2:, :]
    feats = []
    for (l, h), F in zip(bands, filters):
        cf = band_filter(cue, l, h)
        proj = [x @ F for x in cf]
        d = np.log(np.maximum(np.array([np.diag(np.cov(p.T)) for p in proj]), 1e-12))
        feats.append(d)
    return np.concatenate(feats, axis=1)

def timdomain_features(cue, l=4.0, h=30.0, fs_out=50.0, off=0.0):
    """Bandpass + decimate: (N, 8, T/fs_out*...)."""
    if off > 0:
        k2 = int(off * FS)
        cue = cue[:, k2:, :]
    xf = band_filter(cue, l, h)
    r = int(FS / fs_out)
    return xf[:, ::r, :].reshape(xf.shape[0], -1)


In [ ]:
cues, bases, y, sub = [], [], [], []
for f in sorted(glob.glob(os.path.join(train_dir, "*.csv"))):
    c, b, lab = read_epochs(f, labeled=True)
    subj = re.search(r"(S\d+)-", os.path.basename(f)).group(1)
    cues.append(c); bases.append(b); y.append(lab); sub.append(np.full(len(lab), subj))
cues = np.concatenate(cues); bases = np.concatenate(bases)
y = np.concatenate(y); sub = np.concatenate(sub)
subjects = sorted(set(sub))
print("trials:", cues.shape, "move frac:", y.mean())

In [ ]:
bp_all = band_power_features(cues, bases)
print("bp features:", bp_all.shape)

In [ ]:
# v1 features (older set) for the mixed ensemble
def band_powers(x):
    n = x.shape[0]
    xw = (x - x.mean(axis=0)) * np.hanning(n)[:, None]
    spec = np.abs(np.fft.rfft(xw, axis=0)) ** 2
    freqs = np.fft.rfftfreq(n, d=1.0 / FS)
    out = {}
    for b, (lo, hi) in {"delta": (1, 4), "theta": (4, 8), "mu": (8, 12),
                        "beta": (13, 20), "beta2": (20, 30)}.items():
        m = (freqs >= lo) & (freqs < hi)
        out[b] = spec[m, :].mean(axis=0)
    return out

def v1_features(cue, base):
    pc = band_powers(cue)
    feat = []
    for ch in range(8):
        for b in ("delta", "theta", "mu", "beta", "beta2"):
            feat.append(np.log1p(pc[b][ch]))
    if base.shape[0] >= 2 * 250:
        pb = band_powers(base)
        for b in ("mu", "beta", "beta2"):
            for ch in range(8):
                feat.append(np.log((pc[b][ch] + 1e-8) / (pb[b][ch] + 1e-8)))
    else:
        feat.extend([0.0] * 24)
    total = sum(v.mean() for v in pc.values()) + 1e-8
    for b in ("delta", "theta", "mu", "beta", "beta2"):
        feat.append(pc[b].mean() / total)
    half = cue.shape[0] // 2
    p1 = band_powers(cue[:half]); p2 = band_powers(cue[half:2 * half])
    for b in ("mu", "beta"):
        for ch in (1, 2, 3):
            feat.append((p2[b][ch] - p1[b][ch]) / (p1[b][ch] + 1e-8))
    feat.append(pc["mu"][1] + pc["mu"][2] + pc["mu"][3]
                - pc["mu"][4] - pc["mu"][5] - pc["mu"][6] - pc["mu"][7])
    return np.asarray(feat, dtype=np.float32)

v1_all = np.stack([v1_features(cues[i], bases[i]) for i in range(len(y))])
print("v1 features:", v1_all.shape)

In [ ]:
# ---------- CV: subject-grouped 5-fold, mixed ensemble ----------
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(0)
order = rng.permutation(subjects)
folds = np.array_split(order, 5)

def fold_feats(tr, va):
    rts_tr, ref = riemannian_features(cues[tr])
    cspf_tr, f0 = csp_features(cues[tr], y[tr])
    rts_va, _ = riemannian_features(cues[va], ref=ref)
    cspf_va = csp_transform(cues[va], f0)
    return (np.concatenate([bp_all[tr], rts_tr, cspf_tr], 1),
            np.concatenate([bp_all[va], rts_va, cspf_va], 1))

accs = []
oof_ml = np.full(len(y), np.nan)
for fold in folds:
    va = np.isin(sub, list(fold)); tr = ~va
    va_idx = np.flatnonzero(va)
    Ftr, Fva = fold_feats(tr, va)
    Pw3 = np.mean([make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=5000)).fit(Ftr, y[tr]).predict_proba(Fva)[:, 1]
                   for c in (0.1, 1.0)], axis=0)
    Pw3 = np.mean([Pw3, LinearDiscriminantAnalysis(solver="lsqr").fit(Ftr, y[tr]).predict_proba(Fva)[:, 1]], axis=0)
    Pv1 = np.mean([make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=5000)).fit(v1_all[tr], y[tr]).predict_proba(v1_all[va])[:, 1]
                   for c in (0.1, 1.0)], axis=0)
    P = 0.5 * Pw3 + 0.5 * Pv1
    oof_ml[va_idx] = P
    accs.append(((P > 0.5).astype(int) == y[va]).mean())
print("mixed ensemble grouped 5-fold:", np.round(accs, 3), "mean %.4f" % np.mean(accs))
print("oof_ml filled:", float(np.isfinite(oof_ml).mean()), "meanP:", round(float(oof_ml.mean()), 4))

In [ ]:
# ---------- final: fit all models on all data ----------
rts_all, ref_all = riemannian_features(cues)
_, csp_filters_all = csp_features(cues, y)
cspf_all = csp_transform(cues, csp_filters_all)
F_all = np.concatenate([bp_all, rts_all, cspf_all], 1)
print("final w3 features:", F_all.shape)
print("final models trained")

In [ ]:
# ---------- test set (sorted file order) ----------
test_files = sorted(os.listdir(test_dir))
test_cues, test_bases, tsub = [], [], []
for f in test_files:
    c, b, _ = read_epochs(os.path.join(test_dir, f), labeled=False)
    test_cues.append(c); test_bases.append(b)
    tsub.append(np.full(len(c), f))
test_cues = np.concatenate(test_cues); test_bases = np.concatenate(test_bases)
tsub = np.concatenate(tsub)
print("test trials:", test_cues.shape, "files:", len(test_files))

bp_t = band_power_features(test_cues, test_bases)
rts_t, _ = riemannian_features(test_cues, ref=ref_all)
cspf_t = csp_transform(test_cues, csp_filters_all)
F_t = np.concatenate([bp_t, rts_t, cspf_t], 1)
v1_t = np.stack([v1_features(test_cues[i], test_bases[i]) for i in range(len(bp_t))])

Pw3 = np.mean([make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=5000)).fit(F_all, y).predict_proba(F_t)[:, 1]
               for c in (0.1, 1.0)], axis=0)
Pw3 = np.mean([Pw3, LinearDiscriminantAnalysis(solver="lsqr").fit(F_all, y).predict_proba(F_t)[:, 1]], axis=0)
Pv1 = np.mean([make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=5000)).fit(v1_all, y).predict_proba(v1_t)[:, 1]
               for c in (0.1, 1.0)], axis=0)
P = 0.5 * Pw3 + 0.5 * Pv1
pd.DataFrame({"ID": range(len(P)), "file": tsub, "P": P}).to_csv("test_probs.csv", index=False)

# per-file balanced threshold: every session file is a balanced design (k = n//2 move)
pred = np.zeros(len(P), dtype=int)
pred05 = (P > 0.5).astype(int)
print("=== per-file: n, meanP, move@0.5, Ptop10/Pbot10 ===")
for f in test_files:
    m = np.flatnonzero(tsub == f)
    k = len(m) // 2
    pred[m[np.argpartition(-P[m], k)[:k]]] = 1
    pf = P[m]
    print(f, "n=", len(m), "meanP=%.3f" % pf.mean(),
          "move@0.5=", int(pred05[m].sum()),
          "Ptop10=%.3f" % np.sort(pf)[-10:].mean(),
          "Pbot10=%.3f" % np.sort(pf)[:10:].mean())
print("total move: bal=", int(pred.sum()), " @0.5=", int(pred05.sum()), " flip=", int((pred != pred05).sum()))

rows = [{"ID": str(i), "TARGET": "move" if pred[i] == 1 else "rest"} for i in range(len(P))]
df = pd.DataFrame(rows)
df.to_csv("submission.csv", index=False)
print(df.TARGET.value_counts())
print(df.head())
assert len(df) > 0
print("SUBMISSION WRITTEN, rows:", len(df))

# Cross-Subject EEG Motor-Imagery — DL 2-seed (42/1337) x 3-arch CNN ensemble + per-file balanced threshold


In [ ]:
import os, re, time, random, warnings
import numpy as np
import pandas as pd
from scipy import signal
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupKFold
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.__version__)

DATA = "/kaggle/input/competitions/low-cost-motor-imagery-decoding-for-rehab-cross-subject"
TRAIN_DIR = os.path.join(DATA, "Training set")
TEST_DIR = os.path.join(DATA, "Cross participant test set")
CH = ['Fz', 'C3', 'Cz', 'C4', 'PO7', 'Pz', 'PO8', 'Oz']
FS = 250.0
WINDOW_SIZE = 500
WINDOW_STEP = 125

b_notch, a_notch = signal.iirnotch(w0=50.0, Q=30.0, fs=FS)
b_band, a_band = signal.butter(N=4, Wn=[1.0, 45.0], btype='bandpass', fs=FS)

def prep(df):
    raw = df[CH].to_numpy(np.float32)
    x = signal.filtfilt(b_notch, a_notch, raw, axis=0)
    return signal.filtfilt(b_band, a_band, x, axis=0)

def windows_from_trial(trial):
    mu = trial.mean(0, keepdims=True); sd = trial.std(0, keepdims=True)
    z = (trial - mu) / (sd + 1e-8)
    out, p = [], 0
    while p + WINDOW_SIZE <= z.shape[0]:
        out.append(z[p:p+WINDOW_SIZE, :].T.copy())
        p += WINDOW_STEP
    if not out:
        zpad = np.zeros((8, WINDOW_SIZE), np.float32)
        zpad[:, :z.shape[0]] = z.T
        out.append(zpad)
    return out

def load_dir(d, cue_vals):
    X, y, fileid, trialid = [], [], [], []
    trial_counter = 0
    for fi, f in enumerate(sorted(os.listdir(d))):
        df = pd.read_csv(os.path.join(d, f), low_memory=False, dtype={'Marker_val': str})
        m = df['Marker_val']
        trig = m.index[m.notna()].tolist()
        vals = m.iloc[trig].tolist()
        sig = prep(df)
        for i in range(len(trig) - 1):
            if vals[i] in cue_vals:
                trial = sig[trig[i]:trig[i+1], :]
                lab = 1 if vals[i] == 'move' else 0
                for w in windows_from_trial(trial):
                    X.append(w)
                    y.append(lab if cue_vals != ('cue_start',) else -1)
                    fileid.append(fi)
                    trialid.append(trial_counter)
                trial_counter += 1
    X = np.stack(X)[:, None].astype(np.float32)
    return X, np.array(y), np.array(fileid), np.array(trialid)

t0 = time.time()
Xtr, ytr, filtr, trialtr = load_dir(TRAIN_DIR, ('move', 'rest'))
print("train windows:", Xtr.shape, "trials:", int(trialtr.max())+1, round(time.time()-t0, 1), "s")
Xte, yte, filete, trialte = load_dir(TEST_DIR, ('cue_start',))
print("test windows:", Xte.shape, "trials:", int(trialte.max())+1, round(time.time()-t0, 1), "s")
import collections
ct = collections.Counter(zip(filete.tolist(), trialte.tolist()))
print("trials per test file:", {f: len(set(t for ff, t in ct if ff == f)) for f in range(int(filete.max())+1)})


In [ ]:
# ---------- architectures (MultiScaleEEGNet, ShallowConvNet, AttentionEEGNet) ----------
class MultiScaleEEGNet(nn.Module):
    def __init__(self, n_channels=8, n_classes=2, n_samples=500, F1=8, D=2, F2=16):
        super().__init__()
        self.conv_t1 = nn.Conv2d(1, F1, (1, 63), padding=(0, 31), bias=False)
        self.conv_t2 = nn.Conv2d(1, F1, (1, 31), padding=(0, 15), bias=False)
        self.conv_t3 = nn.Conv2d(1, F1, (1, 15), padding=(0, 7), bias=False)
        self.bn_t = nn.BatchNorm2d(F1 * 3)
        self.conv_s = nn.Conv2d(F1 * 3, F1 * 3 * D, (n_channels, 1), groups=F1 * 3, bias=False)
        self.bn_s = nn.BatchNorm2d(F1 * 3 * D)
        self.pool1 = nn.AvgPool2d((1, 4))
        self.drop1 = nn.Dropout(0.25)
        self.conv_sep_depth = nn.Conv2d(F1 * 3 * D, F1 * 3 * D, (1, 15), padding=(0, 7), groups=F1 * 3 * D, bias=False)
        self.conv_sep_point = nn.Conv2d(F1 * 3 * D, F2 * 2, (1, 1), bias=False)
        self.bn_sep = nn.BatchNorm2d(F2 * 2)
        self.pool2 = nn.AvgPool2d((1, 8))
        self.drop2 = nn.Dropout(0.25)
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            x = torch.cat([self.conv_t1(dummy), self.conv_t2(dummy), self.conv_t3(dummy)], dim=1)
            x = self.drop1(self.pool1(F.elu(self.bn_s(self.conv_s(self.bn_t(x))))))
            x = self.conv_sep_point(self.conv_sep_depth(x))
            x = self.drop2(self.pool2(F.elu(self.bn_sep(x))))
            flat_dim = x.view(1, -1).size(1)
        self.classifier = nn.Linear(flat_dim, n_classes)

    def forward(self, x):
        x = torch.cat([self.conv_t1(x), self.conv_t2(x), self.conv_t3(x)], dim=1)
        x = self.bn_t(x)
        x = self.drop1(self.pool1(F.elu(self.bn_s(self.conv_s(x)))))
        x = self.conv_sep_point(self.conv_sep_depth(x))
        x = self.drop2(self.pool2(F.elu(self.bn_sep(x))))
        return self.classifier(x.view(x.size(0), -1))

class ShallowConvNet(nn.Module):
    def __init__(self, n_channels=8, n_classes=2, n_samples=500, F=40):
        super().__init__()
        self.conv_t = nn.Conv2d(1, F, (1, 25), bias=False)
        self.conv_s = nn.Conv2d(F, F, (n_channels, 1), bias=False)
        self.bn = nn.BatchNorm2d(F)
        self.pool = nn.AvgPool2d((1, 75), stride=(1, 15))
        self.drop = nn.Dropout(0.5)
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            x = self.bn(self.conv_s(self.conv_t(dummy)))
            x = self.drop(torch.log(torch.clamp(self.pool(x ** 2), min=1e-6)))
            flat_dim = x.view(1, -1).size(1)
        self.classifier = nn.Linear(flat_dim, n_classes)

    def forward(self, x):
        x = self.bn(self.conv_s(self.conv_t(x)))
        x = torch.log(torch.clamp(self.pool(x ** 2), min=1e-6))
        x = self.drop(x)
        return self.classifier(x.view(x.size(0), -1))

class SqueezeExcitation(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc1 = nn.Linear(channels, max(1, channels // reduction), bias=False)
        self.fc2 = nn.Linear(max(1, channels // reduction), channels, bias=False)
    def forward(self, x):
        b, c, _, t = x.size()
        y = x.mean(dim=-1).view(b, c)
        y = F.relu(self.fc1(y))
        y = F.sigmoid(self.fc2(y)).view(b, c, 1, 1)
        return x * y

class AttentionEEGNet(nn.Module):
    def __init__(self, n_channels=8, n_classes=2, n_samples=500, F1=8, D=2, F2=16):
        super().__init__()
        self.conv_t = nn.Conv2d(1, F1, (1, 63), padding=(0, 31), bias=False)
        self.bn_t = nn.BatchNorm2d(F1)
        self.conv_s = nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False)
        self.bn_s = nn.BatchNorm2d(F1 * D)
        self.se1 = SqueezeExcitation(F1 * D)
        self.pool1 = nn.AvgPool2d((1, 4))
        self.drop1 = nn.Dropout(0.25)
        self.conv_sep_depth = nn.Conv2d(F1 * D, F1 * D, (1, 15), padding=(0, 7), groups=F1 * D, bias=False)
        self.conv_sep_point = nn.Conv2d(F1 * D, F2, (1, 1), bias=False)
        self.bn_sep = nn.BatchNorm2d(F2)
        self.se2 = SqueezeExcitation(F2)
        self.pool2 = nn.AvgPool2d((1, 8))
        self.drop2 = nn.Dropout(0.25)
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            x = self.se1(F.elu(self.bn_s(self.conv_s(self.bn_t(self.conv_t(dummy))))))
            x = self.drop1(self.pool1(x))
            x = self.se2(F.elu(self.bn_sep(self.conv_sep_point(self.conv_sep_depth(x)))))
            x = self.drop2(self.pool2(x))
            flat_dim = x.view(1, -1).size(1)
        self.classifier = nn.Linear(flat_dim, n_classes)

    def forward(self, x):
        x = self.bn_t(self.conv_t(x))
        x = F.elu(self.bn_s(self.conv_s(x)))
        x = self.se1(x)
        x = self.drop1(self.pool1(x))
        x = self.conv_sep_point(self.conv_sep_depth(x))
        x = F.elu(self.bn_sep(x))
        x = self.se2(x)
        x = self.drop2(self.pool2(x))
        return self.classifier(x.view(x.size(0), -1))

print("models defined")

In [ ]:
# ---------- training engine ----------
class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes=2, smoothing=0.05):
        super().__init__()
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.cls = classes
    def forward(self, pred, target):
        pred = pred.log_softmax(dim=-1)
        with torch.no_grad():
            true_dist = torch.zeros_like(pred).fill_(self.smoothing / (self.cls - 1))
            true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)
        return torch.mean(torch.sum(-true_dist * pred, dim=-1))

def make_model(name, **kw):
    return {"MultiScaleEEGNet": MultiScaleEEGNet, "ShallowConvNet": ShallowConvNet,
            "AttentionEEGNet": AttentionEEGNet}[name](**kw)

def train_epochs(model, loader, opt, crit, sched, epochs, Xv, yv):
    Xvt = torch.from_numpy(Xv).to(device)
    yvt = torch.from_numpy(yv).to(device)
    best = -1.0; best_probs = None; best_state = None
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in loader:
            xb = xb.to(device); yb = yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            vp = F.softmax(model(Xvt), dim=1)[:, 1].cpu().numpy()
        acc = ((vp > 0.5).astype(int) == yv).mean()
        if acc > best:
            best = acc
            best_probs = vp.copy()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return best, best_probs

def proba(model, X, bs=256):
    model.eval()
    out = []
    Xt = torch.from_numpy(X).to(device)
    with torch.no_grad():
        for i in range(0, len(Xt), bs):
            out.append(F.softmax(model(Xt[i:i+bs]), dim=1)[:, 1].cpu().numpy())
    return np.concatenate(out)


def set_filebatch(model, on):
    """BN uses test-file batch statistics; dropout stays off."""
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.eval()
    model.train() if on else model.eval()

def proba_filebatch(models, X_by_file):
    """X_by_file: list of (n_win, 1, 8, 500). Returns per-window P stacked."""
    outs = []
    for Xf in X_by_file:
        Xft = torch.from_numpy(Xf).to(device)
        ps = []
        for m in models:
            set_filebatch(m, True)
            with torch.no_grad():
                ps.append(F.softmax(m(Xft), dim=1)[:, 1].cpu().numpy())
            set_filebatch(m, False)
        outs.append(np.stack(ps, 1).mean(1))
    return np.concatenate(outs)

print("engine ready")


In [ ]:
# ---------- CV: 2 seeds x 3 architectures, 5-fold GroupKFold over subjects ----------
EPOCHS = 18
BATCH = 64
LR = 1e-3
WD = 1e-3
N_SPLITS = 5
SEEDS = [42, 1337]
ARCHS = ["MultiScaleEEGNet", "ShallowConvNet", "AttentionEEGNet"]

train_files = sorted(os.listdir(TRAIN_DIR))
file_subj = [re.search(r"(S\d+)-", f).group(1) for f in train_files]
sub_of_win = np.array([file_subj[f] for f in filtr])

n_trials = int(trialtr.max()) + 1
oof_trial = np.full((n_trials, len(SEEDS), len(ARCHS)), np.nan)
fold_acc05, fold_accbal, fold_acc05_fb, fold_accbal_fb = [], [], [], []

gkf = GroupKFold(n_splits=N_SPLITS)
for fold, (tr_i, va_i) in enumerate(gkf.split(np.zeros(len(Xtr)), ytr, groups=sub_of_win)):
    va_trials = np.unique(trialtr[va_i])
    wva = np.isin(trialtr, va_trials)
    wtr = ~wva
    Xtr_, ytr_ = Xtr[wtr], ytr[wtr]
    Xva_ = Xtr[wva]
    yv_t = np.array([ytr[trialtr == k][0] for k in va_trials])
    ds = TensorDataset(torch.from_numpy(Xtr_), torch.from_numpy(ytr_))
    ld = DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=True)
    Xvt = torch.from_numpy(Xva_).to(device)
    final_models_keep = []
    Pw = np.zeros((len(Xva_), len(SEEDS) * len(ARCHS)))
    for si, seed in enumerate(SEEDS):
        for ai, name in enumerate(ARCHS):
            s0 = seed * 1000 + fold * 100 + ai
            random.seed(s0); np.random.seed(s0); torch.manual_seed(s0); torch.cuda.manual_seed_all(s0)
            m = make_model(name).to(device)
            opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
            crit = LabelSmoothingLoss()
            best, best_state = -1.0, None
            for ep in range(1, EPOCHS + 1):
                m.train()
                for xb, yb in ld:
                    xb = xb.to(device); yb = yb.to(device)
                    opt.zero_grad()
                    loss = crit(m(xb), yb)
                    loss.backward()
                    nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                    opt.step()
                sch.step()
                m.eval()
                with torch.no_grad():
                    vp = F.softmax(m(Xvt), dim=1)[:, 1].cpu().numpy()
                acc = ((vp > 0.5).astype(int) == ytr[wva]).mean()
                if acc > best:
                    best, best_state = acc, {k: v.clone() for k, v in m.state_dict().items()}
            m.load_state_dict(best_state)
            m.eval()
            with torch.no_grad():
                Pw[:, si * len(ARCHS) + ai] = F.softmax(m(Xvt), dim=1)[:, 1].cpu().numpy()
            final_models_keep.append(m)
            del opt, sch
            torch.cuda.empty_cache()
            print(f"fold {fold} s{seed} {name} best val win-acc {best:.4f}", flush=True)
    t_all = np.full(n_trials, np.nan)
    per_sa = []
    for j in range(len(SEEDS) * len(ARCHS)):
        per_sa.append(np.array([Pw[trialtr[wva] == k, j].mean() for k in va_trials]))
    oof_trial[va_trials] = np.stack(per_sa, 1).reshape(len(va_trials), len(SEEDS), len(ARCHS))
    t_all[va_trials] = np.stack(per_sa, 1).mean(1)
    acc05 = ((t_all[va_trials] > 0.5).astype(int) == yv_t).mean()
    def file_bal_acc(tvec):
        out = []
        for f in np.unique(filtr[wva]):
            fm = filtr[wva] == f
            tf = np.unique(trialtr[wva][fm])
            if len(tf) < 4: continue
            tfm = np.isin(va_trials, tf)
            k2 = len(tf) // 2
            P = tvec[tf]
            pred = np.zeros(len(tf), int); pred[np.argpartition(-P, k2)[:k2]] = 1
            out.append((pred == yv_t[tfm]).mean())
        return float(np.mean(out))
    accbal = file_bal_acc(t_all)
    # filebatch mode (BN per-validation-file batch stats)
    models_now = list(final_models_keep)
    Xf_list = [Xtr[wva][filtr[wva] == f] for f in np.unique(filtr[wva])]
    Pfb = proba_filebatch(models_now, Xf_list)
    t_fb = np.full(n_trials, np.nan)
    for f in np.unique(filtr[wva]):
        fm = filtr[wva] == f
        tr_w = trialtr[wva][fm]
        tf = np.unique(tr_w)
        t_fb[tf] = [Pfb[fm][tr_w == k].mean() for k in tf]
    acc05_fb = ((t_fb[va_trials] > 0.5).astype(int) == yv_t).mean()
    accbal_fb = file_bal_acc(t_fb)
    fold_acc05.append(acc05); fold_accbal.append(accbal)
    fold_acc05_fb.append(acc05_fb); fold_accbal_fb.append(accbal_fb)
    for m in models_now:
        set_filebatch(m, False)
    print(f"FOLD {fold}: eval-mode 0.5={acc05:.4f} bal={accbal:.4f} | filebatch 0.5={acc05_fb:.4f} bal={accbal_fb:.4f}", flush=True)

print("=== CV trial-level: eval-mode 0.5=%.4f bal=%.4f ===" % (np.mean(fold_acc05), np.mean(fold_accbal)))
print("=== CV trial-level: filebatch 0.5=%.4f bal=%.4f ===" % (np.mean(fold_acc05_fb), np.mean(fold_accbal_fb)))
FB_WINNER = np.mean(fold_accbal_fb) >= np.mean(fold_accbal)
print("using filebatch for submission:", FB_WINNER)
print("folds 0.5:", np.round(fold_acc05, 4))
print("folds bal:", np.round(fold_accbal, 4))
del final_models_keep
import gc; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# ---------- dump OOF (trial-level, per seed/arch) for offline blend studies ----------
first_win = np.array([np.flatnonzero(trialtr == t)[0] for t in range(n_trials)])
tfile_i = filtr[first_win]
tsubj = np.array([file_subj[f] for f in tfile_i])
ty = np.array([ytr[w] for w in first_win])
oof = pd.DataFrame({"trial": np.arange(n_trials), "y": ty, "file": tfile_i, "subj": tsubj})
for si, seed in enumerate(SEEDS):
    for ai, name in enumerate(ARCHS):
        oof[f"p_s{seed}_{name.lower()}"] = oof_trial[:, si, ai]
oof.to_csv("oof_dl.csv", index=False)
print("oof dumped:", oof.shape, "meanP:", round(float(np.nanmean(oof_trial)), 4), flush=True)


In [ ]:
# ---------- final: 2 seeds x 3 architectures on all data, infer test ----------
ds = TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr))
ld = DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=True)
nte = int(trialte.max()) + 1
Xf_test = []
file_trial_idx = []
for f in range(int(filete.max()) + 1):
    fm = filete == f
    Xf_test.append(Xte[fm])
    file_trial_idx.append(np.unique(trialte[fm]))

final_by_seed = {}
Xt = torch.from_numpy(Xte).to(device)
for si, seed in enumerate(SEEDS):
    models = []
    for ai, name in enumerate(ARCHS):
        s0 = seed * 1000 + 777 + ai
        random.seed(s0); np.random.seed(s0); torch.manual_seed(s0); torch.cuda.manual_seed_all(s0)
        m = make_model(name).to(device)
        opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=16, eta_min=1e-5)
        crit = LabelSmoothingLoss()
        for ep in range(1, 17):
            m.train()
            for xb, yb in ld:
                xb = xb.to(device); yb = yb.to(device)
                opt.zero_grad()
                loss = crit(m(xb), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                opt.step()
            sch.step()
        m.eval()
        models.append(m)
        del opt, sch
        torch.cuda.empty_cache()
        print("trained s%d %s" % (seed, name), flush=True)
    final_by_seed[seed] = models

tfile_arr = np.zeros(nte, int)
for f in range(int(filete.max()) + 1):
    tfile_arr[file_trial_idx[f]] = f

def trial_mean(Pw):
    out = np.zeros(nte); cnt = np.zeros(nte)
    np.add.at(out, trialte, Pw); np.add.at(cnt, trialte, 1)
    return out / cnt

def filebatch_trials(Pf):
    out = np.zeros(nte)
    pos = 0
    for f in range(int(filete.max()) + 1):
        tf = file_trial_idx[f]
        tr_w = trialte[filete == f]
        nwin = len(tr_w)
        Pw = Pf[pos:pos + nwin]
        out[tf] = [Pw[tr_w == k].mean() for k in tf]
        pos += nwin
    return out

def arch_trials(m):
    m.eval()
    with torch.no_grad():
        Pw = F.softmax(m(Xt), dim=1)[:, 1].cpu().numpy()
    return trial_mean(Pw)

recs = {}
all_models = []
for seed in SEEDS:
    models = final_by_seed[seed]
    all_models.extend(models)
    with torch.no_grad():
        Pe = np.stack([F.softmax(m(Xt), dim=1)[:, 1].cpu().numpy() for m in models], 1).mean(1)
    Pf = proba_filebatch(models, Xf_test)
    for m in models:
        set_filebatch(m, False)
    recs[f"P{seed}"] = trial_mean(Pe)
    recs[f"P{seed}_eval"] = recs[f"P{seed}"]
    recs[f"P{seed}_fb"] = filebatch_trials(Pf)
    for ai, name in enumerate(ARCHS):
        recs[f"P{seed}_{name.lower()}"] = arch_trials(models[ai])
tP42, tP1337 = recs["P42"], recs["P1337"]
tP = 0.5 * (tP42 + tP1337)
print("test trials:", nte, "| filebatch:", FB_WINNER)
for f in range(int(filete.max()) + 1):
    tf = np.unique(trialte[filete == f])
    print(f"  file {f}: n_trials={len(tf)} meanP42={tP42[tf].mean():.3f} meanP1337={tP1337[tf].mean():.3f} meanP={tP[tf].mean():.3f}")
pd.DataFrame({"trial": np.arange(nte), "file": tfile_arr, "P": tP, **recs}).to_csv("test_probs.csv", index=False)

# placeholder submission from 2-seed DL alone (blend cell overwrites)
pred = np.zeros(nte, int)
ids_by_file = []
for f in range(int(filete.max()) + 1):
    tf = np.unique(trialte[filete == f])
    k2 = len(tf) // 2
    Pf = tP[tf]
    pred[tf[np.argpartition(-Pf, k2)[:k2]]] = 1
    ids_by_file.append(tf)
flat_ids = np.concatenate(ids_by_file)
assert len(flat_ids) == nte and len(np.unique(flat_ids)) == nte
rows = [{"ID": str(i), "TARGET": "move" if pred[flat_ids[i]] == 1 else "rest"} for i in range(len(flat_ids))]
dfsub = pd.DataFrame(rows)
dfsub.to_csv("submission.csv", index=False)
print(dfsub.TARGET.value_counts())
print("SUBMISSION WRITTEN rows:", len(dfsub))


In [ ]:
# ---------- BLEND: ML + 2-seed DL, weight searched on OOF, per-file balanced ----------
assert len(ty) == len(y) and (ty == y).mean() == 1.0, "trial/label alignment mismatch"
def bal_acc_cv(P):
    accs = []
    for f in np.unique(tfile_i):
        m = tfile_i == f
        k2 = int(m.sum()) // 2
        Pm = P[m]
        pred = np.zeros(int(m.sum()), int); pred[np.argpartition(-Pm, k2)[:k2]] = 1
        accs.append(float((pred == ty[m]).mean()))
    return float(np.mean(accs))

o = oof
def ocol(pat):
    cc = [c for c in o.columns if c == pat]
    assert len(cc) == 1, (pat, o.columns)
    return o[cc[0]].to_numpy()
ms42  = ocol("p_s42_multiscaleeegnet"); sc42  = ocol("p_s42_shallowconvnet"); at42  = ocol("p_s42_attentioneegnet")
ms1337= ocol("p_s1337_multiscaleeegnet"); sc1337= ocol("p_s1337_shallowconvnet"); at1337= ocol("p_s1337_attentioneegnet")
m42, m1337 = (ms42+sc42+at42)/3, (ms1337+sc1337+at1337)/3
ocands = {"dl42": m42, "dl1337": m1337, "dl_avg": (m42+m1337)/2,
          "sc42": sc42, "sc1337": sc1337, "sc_avg": (sc42+sc1337)/2,
          "ms42": ms42, "ms1337": ms1337, "ms_avg": (ms42+ms1337)/2,
          "at42": at42, "at1337": at1337, "at_avg": (at42+at1337)/2}
tcands = {"dl42": recs["P42"], "dl1337": recs["P1337"], "dl_avg": tP,
          "sc42": recs["P42_shallowconvnet"], "sc1337": recs["P1337_shallowconvnet"], "sc_avg": (recs["P42_shallowconvnet"]+recs["P1337_shallowconvnet"])/2,
          "ms42": recs["P42_multiscaleeegnet"], "ms1337": recs["P1337_multiscaleeegnet"], "ms_avg": (recs["P42_multiscaleeegnet"]+recs["P1337_multiscaleeegnet"])/2,
          "at42": recs["P42_attentioneegnet"], "at1337": recs["P1337_attentioneegnet"], "at_avg": (recs["P42_attentioneegnet"]+recs["P1337_attentioneegnet"])/2}
results = []
for name, v in ocands.items():
    for w in np.arange(0.0, 1.01, 0.05):
        results.append((round(bal_acc_cv(w * oof_ml + (1 - w) * v), 4), name, round(float(w), 2)))
results.sort(reverse=True)
for r in results[:10]: print(r)
best_acc, best_name, best_w = results[0]
print("BEST blend on OOF (per-file balanced acc):", best_acc, best_name, "w_ml=", best_w)
Pfin = best_w * P + (1 - best_w) * tcands[best_name]
predb = np.zeros(len(Pfin), int)
for f in sorted(set(tsub)):
    m = np.flatnonzero(tsub == f)
    k2 = len(m) // 2
    predb[m[np.argpartition(-Pfin[m], k2)[:k2]]] = 1
pd.DataFrame({"ID": range(len(Pfin)), "file": tsub, "P_ml": P, "P_dl": tcands[best_name], "P_fin": Pfin, "pred": predb}).to_csv("test_probs_blend.csv", index=False)
rows = [{"ID": str(i), "TARGET": "move" if predb[i] == 1 else "rest"} for i in range(len(Pfin))]
dfb = pd.DataFrame(rows)
dfb.to_csv("submission.csv", index=False)
print(dfb.TARGET.value_counts().to_dict())
print("BLEND SUBMISSION WRITTEN rows:", len(dfb))
